## ShoeCo Problem with Backlogging

ShoeCo needs to plan production for the next 4 months. At the beginning of the first month, ShoeCo has 500 pairs of shoes in inventory. The demand forecast for each of the next 4 months is 

Month: |    1 |    2 |    3 |    4 |
 -----:|-----:|-----:|-----:|-----:|
Demand:| 3000 | 5000 | 2000 | 1000 |


ShoeCo has 100 workers currently employed who work 160 hours/month. Each worker is paid \$1500/month. They can also work overtime for up to 20 hours/worker each month. Overtime is paid at \$13/hour.  ShoCo can choose to hire a new worker for \$1600 or fire a worker for \$2000. To produce a pair of shoes, 4 hours of labor and \$15 of raw materials are required. ShoeCo must meet by the end of the 4 months and must pay \$3 for every pair of shoes in inventory at the end of each month. ShoeCo must also pay \$20 for any demand that is backlogged each month. How should ShoeCo produce for the next 4 months if they are trying to minimize their total costs?

### Solution

In [2]:
using JuMP, Clp

d = [3000 5000 2000 1000] # monthly shoe demand
 
m = Model(solver=ClpSolver())

@variable(m, x[1:4] >= 0 ) # shoes produced in month t=1,2,3,4
@variable(m, w[1:5] >= 0 ) # workers employed in month t=0,1,2,3,4
@variable(m, o[1:4] >= 0 ) # overtime hours in month t=1,2,3,4
@variable(m, h[1:4] >= 0 ) # workers hired in month t=1,2,3,4
@variable(m, f[1:4] >= 0 ) # workers fired in month t=1,2,3,4
@variable(m, I[1:5] >= 0 ) # shoes in inventory in month t=0,1,2,3,4
@variable(m, L[1:5] >= 0 ) # shoes leftover in month t=0,1,2,3,4
@variable(m, S[1:5] >= 0 ) # shoes backlogged in month t=0,1,2,3,4

@objective(m, Min, 15*sum(x) + 13*sum(o) + 1600*sum(h) + 2000*sum(f)
    + 1500*sum(w) + 3*sum(L) + 20*sum(S))

@constraint(m, production[t in 1:4], 4*x[t] <= 160*w[t] + o[t])
@constraint(m, overtime[t in 1:4], o[t] <= 20*w[t])
@constraint(m, inv_bal[t in 1:4], I[t] + x[t] == d[t] + I[t+1])
@constraint(m, inv_ident[t in 1:5], I[t] == L[t] - S[t])
@constraint(m, I[1] == 500)
@constraint(m, work_bal[t in 1:4], w[t] - f[t] + h[t] == w[t+1])
@constraint(m, w[1] == 100)

solve(m)

println("Build ", Array(getvalue(x')), " shoes each month")
println("Use ", Array(getvalue(w')), " workers each month")
println("Use ", Array(getvalue(o')), " overtime hours each month")
println("Inventory: ", Array(getvalue(I')))
println("Cost: ", getobjectivevalue(m))

Build [4000.0 3500.0 2000.0 1000.0] shoes each month
Use [100.0 87.5 50.0 25.0 25.0] workers each month
Use [0.0 0.0 0.0 0.0] overtime hours each month
Inventory: [500.0 1500.0 0.0 0.0 0.0]
Cost: 744750.0
